In [ ]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 라이브러리 설치
!pip install numpy opencv-python insightface onnxruntime-gpu


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 439.5/439.5 kB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 52.4 MB/s eta 0:00:00
  Created wheel for insightface: filename=insightface-0.7.3-cp312-cp312-linux_x86_64.whl size=1071489 sha256=186494b20fea39f3405f8a0c3ff0de308888f6ad17560a5d36e8ae5834dd3b7f
  Stored in directory: /root/.cache/pip/wheels/73/3c/e2/6d4815e8a8b33a2006554d65ce0d1f973e768f4c7a222fa675
Successfully built insightface


# 계층 구조 없는 단일 폴더 내에 모든 이미지 존재

In [ ]:
import os
import glob
import cv2
import numpy as np
from insightface.app import FaceAnalysis
from insightface.utils import face_align

# ==========================================
# 1. 경로 설정 (사용자 환경에 맞게 수정하세요)
# ==========================================
INPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인'  # 원본 이미지 폴더 경로
OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리' # 저장할 폴더 경로

# 출력 폴더가 존재하지 않으면 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 지원할 이미지 확장자 정의
IMAGE_EXTENSIONS = ('*.jpg', '*.jpeg', '*.png', '*.webp', '*.bmp')

# ==========================================
# 2. InsightFace 모델 준비
# ==========================================
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

# ==========================================
# 3. 이미지 일괄 처리 루프
# ==========================================
# 폴더 내 모든 이미지 경로 가져오기
image_paths = []
for ext in IMAGE_EXTENSIONS:
    # 대소문자 구분 없이 검색하기 위해 glob 사용
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, ext)))
    image_paths.extend(glob.glob(os.path.join(INPUT_DIR, ext.upper())))

# 중복 제거 및 정렬
image_paths = sorted(list(set(image_paths)))
print(f"총 {len(image_paths)}개의 이미지를 찾았습니다. 전처리를 시작합니다.\n")

for img_idx, img_path in enumerate(image_paths):
    # 파일명만 추출 (예: 'photo.jpg')
    base_name = os.path.basename(img_path)
    # 파일명과 확장자 분리 (예: 'photo', '.jpg')
    file_name, _ = os.path.splitext(base_name)

    # 이미지 로드
    img = cv2.imread(img_path)
    if img is None:
        print(f"[{img_idx+1}/{len(image_paths)}] 이미지를 불러올 수 없습니다: {base_name}")
        continue

    # 얼굴 검출
    faces = app.get(img)
    print(f"[{img_idx+1}/{len(image_paths)}] {base_name} - 검출된 얼굴 수: {len(faces)}")

    # 검출된 얼굴별로 정렬 및 저장
    for face_idx, face in enumerate(faces):
        # face_align을 사용하여 112x112로 정렬된 이미지 추출
        aimg = face_align.norm_crop(img, landmark=face.kps, image_size=112)

        # 저장할 파일명 정의 (예: photo_face_0.jpg, photo_face_1.jpg ...)
        save_name = f"{file_name}_face_{face_idx}.jpg"
        save_path = os.path.join(OUTPUT_DIR, save_name)

        # 정렬된 얼굴 이미지 저장
        cv2.imwrite(save_path, aimg)

print("\n모든 이미지의 얼굴 전처리 및 저장이 완료되었습니다!")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
총 0개의 이미지를 찾

# 한 폴더 내 여러 폴더 있고 그 안에 이미지 존재

In [ ]:
import os
import cv2
import numpy as np
from insightface.app import FaceAnalysis
from insightface.utils import face_align
import unicodedata

# ==========================================
# 1. 경로 설정
# ==========================================
INPUT_DIR = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인'
OUTPUT_DIR = r'/content/drive/MyDrive/Colab Notebooks/SafeAI/팀플/데이터셋/한국 연예인_전처리'

os.makedirs(OUTPUT_DIR, exist_ok=True)
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.webp', '.bmp')

# ==========================================
# 2. InsightFace 모델 준비
# ==========================================
app = FaceAnalysis(name='buffalo_l')
app.prepare(ctx_id=0, det_size=(640, 640))

print("🚀 데이터 전처리를 시작합니다...\n" + "="*50)

# INPUT_DIR 내의 인물(ID) 폴더 리스트만 가져오기
identity_folders = [f for f in os.listdir(INPUT_DIR) if os.path.isdir(os.path.join(INPUT_DIR, f))]

total_frontal = 0
total_profile = 0

# ==========================================
# 3. 인물(폴더) 단위 처리 루프
# ==========================================
for identity in identity_folders:
    safe_identity = unicodedata.normalize('NFC', identity)
    src_identity_path = os.path.join(INPUT_DIR, identity)
    out_identity_dir = os.path.join(OUTPUT_DIR, safe_identity)

    os.makedirs(out_identity_dir, exist_ok=True)

    # 해당 인물 폴더 안의 파일 리스트
    all_files = os.listdir(src_identity_path)

    frontal_saved_cnt = 0
    profile_saved_cnt = 0

    for file in all_files:
        if not file.lower().endswith(IMAGE_EXTENSIONS):
            continue

        safe_file_name = unicodedata.normalize('NFC', file)
        img_path = os.path.join(src_identity_path, file)

        # 이미지 로드 (OpenCV 한글 경로 에러 방지)
        try:
            img_array = np.fromfile(img_path, np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
        except:
            continue

        if img is None:
            continue

        # 얼굴 검출
        faces = app.get(img)
        if len(faces) == 0:
            continue
        elif len(faces) != 1:
            continue

        file_base, _ = os.path.splitext(safe_file_name)

        # 검출된 얼굴 크롭 및 저장
        for face_idx, face in enumerate(faces):
            aimg = face_align.norm_crop(img, landmark=face.kps, image_size=112)

            save_name = f"{file_base}_face_{face_idx}.jpg"
            save_path = os.path.join(out_identity_dir, save_name)

            is_success, encoded_img = cv2.imencode('.jpg', aimg)
            if is_success:
                with open(save_path, mode='w+b') as f:
                    encoded_img.tofile(f)

                # 성공적으로 저장된 경우에만 카운트 증가
                if '옆모습' in safe_file_name:
                    profile_saved_cnt += 1
                else:
                    frontal_saved_cnt += 1  # '옆모습'이 없으면 무조건 정면으로 간주

    # 한 인물(폴더)의 루프가 끝나면 결과 요약 출력
    print(f"👤 [{safe_identity}] 전처리 완료 ➔ 정면: {frontal_saved_cnt}장, 옆모습: {profile_saved_cnt}장 저장됨")

    total_frontal += frontal_saved_cnt
    total_profile += profile_saved_cnt

print("="*50)
print(f"🎉 전처리 모두 완료!")
print(f"총 누적 저장량 ➔ 정면: {total_frontal}장, 옆모습: {total_profile}장")
print(f"📍 저장 경로: {OUTPUT_DIR}")

Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['None', 3, 112, 112] 127.5 127.5
set det-size: (640, 640)
🚀 데이터 전처리를 시